Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [2]:
raw_data = pd.read_csv("netflix_titles.csv")

In [4]:
raw_data.head(1)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."


In [11]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [14]:
cleaning_log={}

Initial inspection

In [15]:
cleaning_log["initial_shape"]=raw_data.shape
cleaning_log["initial_missingvalues"]=raw_data.isnull().sum().to_dict()

Removeing Duplicated

In [34]:
duplicates = raw_data.duplicated().sum()
raw_data=raw_data.drop_duplicates()
cleaning_log["duplicates_removed"]=int(duplicates)

Parseing Dates Safely

In [18]:
raw_data["date_added"]=pd.to_datetime(raw_data["date_added"],errors="coerce")
cleaning_log["date_parsing_null's "]=int(raw_data["date_added"].isna().sum())

Fixing Numeric Colums

In [20]:
raw_data["release_year"]=pd.to_numeric(raw_data["release_year"],errors="coerce")

Splitting the colums by data type

In [21]:
num_col=raw_data.select_dtypes(include=["int64","float64"]).columns
cat_col=raw_data.select_dtypes(include=["object"]).columns


Adding the imputation

In [22]:
si_impute = SimpleImputer(strategy="median")
raw_data[num_col]=si_impute.fit_transform(raw_data[num_col])

In [25]:
csi_impute = SimpleImputer(strategy="most_frequent")
raw_data[cat_col]=csi_impute.fit_transform(raw_data[cat_col])

In [26]:
cleaning_log["numeric_imputation"] = "median"
cleaning_log["categorical_imputation"] = "most_frequent"

In [27]:
raw_data["duration_int"] = raw_data["duration"].str.extract(r'(\d+)').astype(float)

In [29]:
cleaning_log["final_missing"] = raw_data.isnull().sum().to_dict()
cleaning_log["final_shape"] = raw_data.shape
cleaning_log["final_dtypes"] = raw_data.dtypes.astype(str).to_dict()

Saving the file

In [30]:
raw_data.to_csv("netflix_titles_cleaned.csv", index=False)

In [40]:
log_file = pd.DataFrame({
    "metric": cleaning_log.keys(),
    "value": [str(v) for v in cleaning_log.values()]
})
log_file.to_csv("cleaning_log_advance.csv",index=False)
